# Chapter 16: time series I

Generate IID and stationary AR(1) monthly series with equal theoretical mean and variance. Compare ARIMA(1,0,0) predictions with an expanding training-mean benchmark using shared time-series functions. A frozen ACF reference image is displayed, not regenerated. The French description is disabled and does not establish PIT availability or trading performance.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch16_time_series_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
from base64 import b64encode
from IPython.display import HTML, display

from quant_course.time_series import (
    ar1_theoretical_moments,
    fit_arima,
    forecast_arima,
    ljung_box_diagnostic,
    rolling_origin_forecast,
    sample_acf_pacf,
    simulate_stationary_pair,
    split_monthly_series,
    training_mean_forecast,
    validate_monthly_series,
)

print("project root:", ROOT)

## Equal moments, different dependence

Construct 480 months with mean 0.005, marginal standard deviation 0.04 and phi=0.7. Set innovation SD to marginal SD times sqrt(1−phi²). Finite-sample estimates need not agree exactly. The frozen chart contrasts zero theoretical IID autocorrelation with geometric AR(1) decay.

In [ ]:
pair = simulate_stationary_pair(n=480, seed=20261016, mean=0.005, marginal_sd=0.04, phi=0.7, start="1980-01")
moments = ar1_theoretical_moments(mean=0.005, phi=0.7, innovation_sd=0.04 * np.sqrt(1 - 0.7**2), max_lag=3)
print("AR(1) theoretical mean / variance / ACF:", moments.mean, moments.variance, moments.autocorrelations)
for column in ("iid", "ar1"):
    summary = sample_acf_pacf(pair[column], lags=12)
    print(column, "sample mean", pair[column].mean(), "sample sd", pair[column].std(ddof=1), "ACF(1)", summary.loc[1, "acf"])

figure_path = ROOT / "fixtures/figures/a4/ch16_stationarity_acf.png"
assert figure_path.is_file(), f"run code/ch16_time_series.py first to generate {figure_path}"
figure_data = b64encode(figure_path.read_bytes()).decode("ascii")
figure_alt = "四格图比较月度收益百分比路径和按月滞后的自相关：Gaussian IID 的理论 ACF 为零，phi 为 0.70 的平稳 AR(1) 理论 ACF 按 0.70 的幂衰减。"
display(HTML(f'<img alt="{figure_alt}" src="data:image/png;base64,{figure_data}" style="max-width: 100%; height: auto;">'))

## Fit a prefix and predict the next month

The first 360 months form the initial training window. Compute a three-step forecast and expanding one-step predictions over the remainder; compare RMSE against the historical training mean. Ljung–Box tests joint linear residual autocorrelation through lag 12. A high p-value neither proves model correctness nor rules out volatility clustering. Repeated fitting costs more than hand arithmetic.

In [ ]:
train, test = split_monthly_series(pair["ar1"], train_size=360)
forecast = forecast_arima(train, steps=3, order=(1, 0, 0), alpha=0.05)
fitted = fit_arima(train, order=(1, 0, 0))
ljung_box = ljung_box_diagnostic(fitted.residuals, model_df=1, lag=12)
rolling = rolling_origin_forecast(pair["ar1"], initial_train_size=360, order=(1, 0, 0))
rolling_mean = np.array([pair["ar1"].iloc[:i].mean() for i in range(360, len(pair))])
rmse_ar = np.sqrt(np.mean((rolling["actual"].to_numpy() - rolling["forecast"].to_numpy()) ** 2))
rmse_mean = np.sqrt(np.mean((rolling["actual"].to_numpy() - rolling_mean) ** 2))
print("training through", train.index[-1], "test begins", test.index[0])
print(forecast)
print("Ljung-Box residual check at lag 12:")
print(ljung_box)
print("simulated expanding-window RMSE ARIMA / train mean:", rmse_ar, rmse_mean)
print("first target and its information origin:", rolling.iloc[0]["origin_period"], "->", rolling.index[0])

## Disabled French description

Historical processed data contain value-weighted industry returns already converted to decimals. YYYYMM becomes a monthly PeriodIndex, not a release timestamp. No sequence of historical available_at values is provided, so the current-vintage description cannot establish real-time availability or forecast skill.

In [ ]:
if RUN_FRENCH:
    french_path = ROOT / "data/processed/french_10_industries_monthly.csv"
    french_table = pd.read_csv(french_path, dtype={"month": "string"})
    french_index = pd.PeriodIndex(french_table.pop("month"), freq="M", name="month")
    no_dur = validate_monthly_series(pd.Series(french_table["NoDur"].to_numpy(dtype=float), index=french_index, name="NoDur"), min_observations=24)
    french_acf = sample_acf_pacf(no_dur, lags=12)
    print("French current snapshot, decimal monthly returns:", no_dur.index[0], "to", no_dur.index[-1], "n=", len(no_dur))
    print("NoDur mean / standard deviation:", no_dur.mean(), no_dur.std(ddof=1))
    print("descriptive ACF(1), ACF(12):", french_acf.loc[1, "acf"], french_acf.loc[12, "acf"])
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## Research boundary

Model order, windows, estimation, intervals and benchmarks are design choices. Trading also requires availability, signal and fill times, matched holding intervals and costs; none are defined here for French data. ADF, VAR and cointegration belong to Chapter 17.